**Testne funkcije**

In [1]:
import pprint as pp

def test(func, *ulazi_ocekivano):
    *ulazi, ocekivano = ulazi_ocekivano
    izlazi = func(*ulazi)
    print(f"{func.__name__}({', '.join(map(repr, ulazi))}) ⮕ {izlazi!r} "
          f"{'👍' if izlazi == ocekivano else f'❌ != {ocekivano!r}'}")

def ttest(fun, *args_expected):
    def pprint(*args, as_repr=True, offset=0, sep=" ", end="\n"):
        formatted_args = []
        for arg in args:
            if as_repr:
                if isinstance(arg, (list, tuple)) and len(arg) > 1 and all(isinstance(row, (list, tuple)) for row in arg):
                    width = max(len(repr(row)) for row in arg) + 2
                    formatted_args.append([line.ljust(width, " ") for line in pp.pformat(arg, width=width, sort_dicts=False).split("\n")])
                elif isinstance(arg, dict) and len(arg) > 1:
                    width = max(len(repr(item)) for item in arg.items())
                    formatted_args.append([line.ljust(width, " ") for line in pp.pformat(arg, width=width, sort_dicts=False).split("\n")])
                else: formatted_args.append(repr(arg))
            else: formatted_args.append(str(arg))

        pos = 0
        for i, arg in enumerate(formatted_args):
            if isinstance(arg, str):
                print(arg, end=""); pos += len(arg)
            else:
                print(arg[0]); pos += offset; offset = 0
                for line in arg[1:-1]: print(" " * pos + line)
                print(" " * pos + arg[-1], end=""); pos += len(arg[-1])
            if i < len(formatted_args) - 1: print(sep, end=""); pos += len(sep)
        print(end=end); return pos + offset + len(str(end))

    *args, expected = args_expected
    offset = pprint(f" {fun.__name__}(", as_repr=False, offset=0, sep="", end="")
    offset = pprint(*args, as_repr=True, offset=offset, sep=", ", end="")
    offset = pprint(") ➤ ", as_repr=False, offset=offset, sep="", end="")
    izlazi = fun(*args)
    offset = pprint(izlazi, as_repr=True, offset=offset, sep=", ", end="")

    if izlazi == expected: pprint(" 👍", offset=offset, as_repr=False, end="")
    else:
        offset = pprint(" ❌ != ", as_repr=False, offset=offset, end="")
        pprint(expected, as_repr=True, offset=offset, end="")
    print("\n")

# bazan

Za dani broj u brojevnom sustavu s bazom `2 <= n <= 36` vrati vrijednost tog broja u dekadskom brojevnom sustavu.
Napomena: za brojevni sustav s bazom `> 10` koriste se mala slova za znamenke koje odgovaraju dekadskim brojevima i to:
```
a = 10
b = 11
c = 12
...
z = 36
```  

Primjer: za broj `b3a8` u bazi `12` se dobije `19568` u dekadskoj bazi

In [2]:
def bazan(b, n):
    b10 = 0
    p = len(b) - 1
    for z in b.lower():
        if z.isalpha():
            z10 = 10 + ord(z) - ord("a")
        else:
            z10 = int(z)

        b10 += z10 * n ** p
        p -= 1
    return b10


test(bazan, "b3a8", 12, 19568)
test(bazan, "ff", 16, 255)
test(bazan, "1001", 2, 9)

bazan('b3a8', 12) ⮕ 19568 👍
bazan('ff', 16) ⮕ 255 👍
bazan('1001', 2) ⮕ 9 👍


# cak

Za dani neprazni string `znakovi` i listu cijelih brojeva `koraci = [k1, k2, k3, ...]`
napravi program koji će generirati novi string na sljedeći način:
1. početna pozicija u stringu `znakovi` je indeks `0`
2. zatim se iterativno prolazi kroz listu `koraci`:
  * za svaki korak `k`, trenutni indeks se pomiče za `k` mjesta u odnosu na prethodnu poziciju
  * ako pomak vodi izvan granica striga, pomak se "odbija" od ruba i kretanje se nastavlja iz važeće pozicije
3. nakon svakog koraka u rezultat se dodaje znak na novoj poziciji 
4. postupak se nastavlja dok se ne obrade svi elementi liste `koraci`

In [3]:
def cak(znakovi, koraci):
    n = len(znakovi)

    if n == 1:
        return znakovi[0] * len(koraci)

    i = 0
    rezultat = ""

    for k in koraci:
        s = 1 if k > 0 else -1
        for _ in range(abs(k)):
            if i == 0 and s == -1:
                i += 1
                s = 1
            elif i == n - 1 and s == 1:
                i -= 1
                s = -1
            else:
                i += s

        rezultat += znakovi[i]
    return rezultat

test(cak, "abcde", [1, 2], "bd")
test(cak, "abcdef", [1, 2, 3], "bde")
test(cak, "abcdef", [1, 2, 3, -4, 2], "bdeac")

test(cak, "abcdef", [-1], "b")
test(cak, "abcdef", [-2], "c")
test(cak, "abcdef", [3, -2], "db")

test(cak, "abcde", [10], "c")
test(cak, "abcde", [12], "e")
test(cak, "abcde", [200], "a")

test(cak, "a", [1, 2, 3], "aaa")
test(cak, "x", [-100, 0, 100], "xxx")
test(cak, "xywz", [1, -100, 0, 100], "yzzy")


cak('abcde', [1, 2]) ⮕ 'bd' 👍
cak('abcdef', [1, 2, 3]) ⮕ 'bde' 👍
cak('abcdef', [1, 2, 3, -4, 2]) ⮕ 'bdeac' 👍
cak('abcdef', [-1]) ⮕ 'b' 👍
cak('abcdef', [-2]) ⮕ 'c' 👍
cak('abcdef', [3, -2]) ⮕ 'db' 👍
cak('abcde', [10]) ⮕ 'c' 👍
cak('abcde', [12]) ⮕ 'e' 👍
cak('abcde', [200]) ⮕ 'a' 👍
cak('a', [1, 2, 3]) ⮕ 'aaa' 👍
cak('x', [-100, 0, 100]) ⮕ 'xxx' 👍
cak('xywz', [1, -100, 0, 100]) ⮕ 'yzzy' 👍


# okvir

Za zadanu veličinu kvadratne matrice `n` popuni matricu `n x n` povećavajući vrijednosti po "okvirima". 
* vanjski okvir = 1
* sljedeći okvir = 2
* zatim 3, itd.

Primjer, za `n = 5`, matrica izgleda ovako:
```
1 1 1 1 1
1 2 2 2 1
1 2 3 2 1
1 2 2 2 1
1 1 1 1 1
```

In [4]:
def okvir(n):
    m = []

    for i in range(n):
        red = []
        for j in range(n):
            top = i
            left = j
            bottom = n - 1 - i
            right = n - 1 - j

            val = min(top, left, bottom, right) + 1
            red.append(val)
        m.append(red)
    return m

ttest(okvir, 1, [[1]])
ttest(okvir, 2, [[1, 1], 
                 [1, 1]])
ttest(okvir, 3, [[1, 1, 1], 
                 [1, 2, 1], 
                 [1, 1, 1]])
ttest(okvir, 4, [[1, 1, 1, 1], 
                 [1, 2, 2, 1], 
                 [1, 2, 2, 1], 
                 [1, 1, 1, 1]])
ttest(okvir, 5, [[1, 1, 1, 1, 1], 
                 [1, 2, 2, 2, 1], 
                 [1, 2, 3, 2, 1], 
                 [1, 2, 2, 2, 1], 
                 [1, 1, 1, 1, 1]])

 okvir(1) ➤ [[1]] 👍

 okvir(2) ➤ [[1, 1],
             [1, 1]] 👍

 okvir(3) ➤ [[1, 1, 1],
             [1, 2, 1],
             [1, 1, 1]] 👍

 okvir(4) ➤ [[1, 1, 1, 1],
             [1, 2, 2, 1],
             [1, 2, 2, 1],
             [1, 1, 1, 1]] 👍

 okvir(5) ➤ [[1, 1, 1, 1, 1],
             [1, 2, 2, 2, 1],
             [1, 2, 3, 2, 1],
             [1, 2, 2, 2, 1],
             [1, 1, 1, 1, 1]] 👍



# pong

Napisati funkciju koja prima tri pozitivna cijela broja:  
* `r > 0` broj redaka matrice 
* `s > 0` broj stupaca matrice 
* `d > 0` duljinu niza brojeva koje treba upisati 

Funkcija treba:
1. Kreirati nul-matricu dimenzije `r x s`
2. Postaviti početnu poziciju u gornji lijevi kut matrice
3. U matricu redom upisivati brojeve od `1` do `d` na sljedeći način:
    * Kretanje po matrici odvija se dijagonalno (istovremeno u smjeru redaka i stupaca)
    * Kada se dosegne rub matrice (gornji, donji, lijevi, desni), smjer kretanja se mijenja tako da se "odbija" od ruba
    * Upis se nastavlja sve dok se ne upišu svi brojevi od `1` do `d`.
    * Ako se tijekom kretanja ponovno dođe na već popunjeno polje, vrijednost se prepisuje novom.

Funkcija na kraju vraća dobivenu matricu.

Na primjer, za `r=9, s=6, d=15` se dobije
```
 1  0  0  0  0  0 
 0  2  0  0  0  0 
 0  0  3  0 15  0
 0  0  0 14  0  0
 0  0 13  0  5  0
 0 12  0  0  0  6
11  0  0  0  7  0
 0 10  0  8  0  0
 0  0  9  0  0  0
```

broj `14` je prebrisao prethodni broj `4`

In [5]:
def pong(r, s, d):
    a = [[0 for _ in range(s)] for _ in range(r)]
    x = y = 0
    dx, dy = 1 if r > 1 else 0, 1 if s > 1 else 0

    for b in range(1, d + 1):
        a[x][y] = b

        if dx == 0:
            pass
        elif dx == 1:
            if 0 < x == r - 1:
                dx = -1
        else:
            if x == 0:
                dx = 1

        if dy == 0:
            pass
        elif dy == 1:
            if y == s - 1:
                dy = -1
        else:
            if y == 0:
                dy = 1

        x += dx
        y += dy

        
    return a


ttest(pong, 1, 1, 5, [[5]])
ttest(pong, 1, 2, 5, [[5, 4]])
ttest(pong, 2, 1, 5, [[5],
                        [4]])
ttest(pong, 2, 2, 5, [[5, 0],
                        [0, 4]])
ttest(pong, 2, 3, 5, [[5, 0, 3],
                        [0, 4, 0]])
ttest(pong, 4, 5, 7, [[1, 0, 7, 0, 0],
                        [0, 2, 0, 6, 0],
                        [0, 0, 3, 0, 5],
                        [0, 0, 0, 4, 0]])
ttest(pong, 4, 5, 10, [[1, 0, 7, 0, 0], 
                        [0, 8, 0, 6, 0], 
                        [9, 0, 3, 0, 5], 
                        [0, 10, 0, 4, 0]])
ttest(pong, 9, 6, 15, [[ 1,  0,  0,  0,  0, 0], 
                        [ 0,  2,  0,  0,  0, 0], 
                        [ 0,  0,  3,  0, 15, 0],
                        [ 0,  0,  0, 14,  0, 0],
                        [ 0,  0, 13,  0,  5, 0],
                        [ 0, 12,  0,  0,  0, 6],
                        [11,  0,  0,  0,  7, 0],
                        [ 0, 10,  0,  8,  0, 0],
                        [ 0,  0,  9,  0,  0, 0]])


 pong(1, 1, 5) ➤ [[5]] 👍

 pong(1, 2, 5) ➤ [[5, 4]] 👍

 pong(2, 1, 5) ➤ [[5],
                  [4]] 👍

 pong(2, 2, 5) ➤ [[5, 0],
                  [0, 4]] 👍

 pong(2, 3, 5) ➤ [[5, 0, 3],
                  [0, 4, 0]] 👍

 pong(4, 5, 7) ➤ [[1, 0, 7, 0, 0],
                  [0, 2, 0, 6, 0],
                  [0, 0, 3, 0, 5],
                  [0, 0, 0, 4, 0]] 👍

 pong(4, 5, 10) ➤ [[1, 0, 7, 0, 0], 
                   [0, 8, 0, 6, 0], 
                   [9, 0, 3, 0, 5], 
                   [0, 10, 0, 4, 0]] 👍

 pong(9, 6, 15) ➤ [[1, 0, 0, 0, 0, 0], 
                   [0, 2, 0, 0, 0, 0], 
                   [0, 0, 3, 0, 15, 0],
                   [0, 0, 0, 14, 0, 0],
                   [0, 0, 13, 0, 5, 0],
                   [0, 12, 0, 0, 0, 6],
                   [11, 0, 0, 0, 7, 0],
                   [0, 10, 0, 8, 0, 0],
                   [0, 0, 9, 0, 0, 0]]  👍

